# SDTM CT NCIt Enrich — Enrich Test Codes with NCIt and UMLS

Reads CDISC CT submission values from SDTM_CT_Extract and enriches each test code
with data from NCI EVS sources:

- **Thesaurus.FLAT.zip** — NCIt tab-delimited flat file: preferred term, synonyms, definition
- **nci_code_cui_map.dat** — NCIt-to-UMLS CUI mapping (Standard CUI and NCI Metathesaurus CUI)

## Pipeline Position

```
sdtm-test-codes/
├── notebooks/
│   ├── SDTM_CT_Extract.ipynb          ← upstream
│   └── SDTM_CT_NCIt_Enrich.ipynb      ← this notebook
├── downloads/
│   ├── SDTM_Terminology.txt           ← NCI EVS (shared with Extract)
│   ├── Thesaurus.FLAT.zip / .txt       ← NCIt flat file
│   └── nci_code_cui_map.dat           ← NCIt-to-UMLS mapping
├── interim/
│   └── SDTM_CT_Extract.csv            ← input from Extract
├── machine_actionable/
│   └── SDTM_Test_Identity.xlsx        ← output (reference file)
└── reports/
```

## What This Notebook Produces

| Source | What it provides | Output columns |
|---|---|---|
| SDTM Terminology txt | Preferred term (CDISC-scoped) | NCIt_Preferred_Term |
| Thesaurus.FLAT | Synonyms (undifferentiated), definition | NCIt_Synonyms, NCIt_Definition |
| nci_code_cui_map.dat | UMLS CUI (Standard) and NCI Metathesaurus CUI | UMLS_CUI, NCIm_CUI |

## About the FLAT File

NCI publishes NCIt in three formats. This notebook uses the flat file
(`Thesaurus.FLAT.zip`, ~15 MB compressed) rather than the OWL
(`Thesaurus.OWL.zip`, ~120 MB compressed) because:

- Parse time: seconds (pandas) vs 10–20 minutes (rdflib OWL)
- No additional dependencies — no rdflib required
- Contains everything needed: synonyms, definitions

**Known limitation of FLAT vs OWL:** The FLAT file provides synonyms as a
plain pipe-delimited list with no type (SY/AB/CN/PT) or source (CDISC/NCI/LOINC)
annotation. All synonyms are undifferentiated. The OWL is the upgrade path
if per-synonym provenance becomes required.

## Design Decision: NCIt Categories Not Included

This file deliberately excludes NCIt hierarchical categories (semantic types,
parent concepts). NCIt categories are too broad for test identity — "Laboratory
Procedure" groups hundreds of unrelated tests, and tests measuring the same
analyte (e.g. all glucose variants) have no explicit relationship in NCIt's
hierarchy. The identity of a test code is established through its NCIt concept,
preferred term, synonyms, and cross-references — not through categorical
placement.

In [ ]:
import pandas as pd
import requests
import re
import zipfile
from pathlib import Path
from datetime import datetime
from collections import Counter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('=' * 70)
print('SDTM CT NCIt ENRICH — TEST CODES WITH NCIt AND UMLS')
print('=' * 70)
print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 1. Configuration

In [ ]:
USE_CACHE = True

BASE_DIR = Path.cwd().parent
DOWNLOADS_DIR = BASE_DIR / 'downloads'
INTERIM_DIR   = BASE_DIR / 'interim'
MA_DIR        = BASE_DIR / 'machine_actionable'
REPORTS_DIR   = BASE_DIR / 'reports'

for d in [DOWNLOADS_DIR, MA_DIR, REPORTS_DIR]:
    d.mkdir(exist_ok=True)

# Input from SDTM_CT_Extract
INPUT_CSV = INTERIM_DIR / 'SDTM_CT_Extract.csv'

# NCIt FLAT file — synonyms, definition
# ~15 MB compressed, seconds to parse. Versionless URL = always current release.
NCIT_FLAT_ZIP  = DOWNLOADS_DIR / 'Thesaurus.FLAT.zip'
NCIT_FLAT_FILE = DOWNLOADS_DIR / 'Thesaurus.txt'
NCIT_FLAT_URL  = 'https://evs.nci.nih.gov/ftp1/NCI_Thesaurus/Thesaurus.FLAT.zip'

# NCIt-to-UMLS CUI mapping — UMLS CUI not in FLAT file, needs separate file
NCIT_CUI_FILE = DOWNLOADS_DIR / 'nci_code_cui_map.dat'
NCIT_CUI_URL  = 'https://evs.nci.nih.gov/ftp1/NCI_Thesaurus/nci_code_cui_map.dat'

# SDTM Terminology — for preferred term lookup (CDISC-scoped)
SDTM_FILE = DOWNLOADS_DIR / 'SDTM_Terminology.txt'
SDTM_URL  = 'https://evs.nci.nih.gov/ftp1/CDISC/SDTM/SDTM%20Terminology.txt'

# Output
OUTPUT_FILE = MA_DIR / 'SDTM_Test_Identity.xlsx'

print(f'Input:   {INPUT_CSV}')
print(f'Output:  {OUTPUT_FILE}')

## 2. Load Input Data

Read the all-domain interim CSV from SDTM_CT_Extract. Deduplicate to unique
NCIt codes by aggregating domain membership.

In [ ]:
df_raw = pd.read_csv(INPUT_CSV, dtype=str).fillna('')
print(f'Loaded {len(df_raw):,} rows from {INPUT_CSV.name}')
print(f'Columns: {df_raw.columns.tolist()}')
print(f'Domains: {sorted(df_raw["domain"].unique())}')

df = df_raw.groupby('NCIt_code').agg(
    TESTCD=('TESTCD', 'first'),
    TEST=('TEST', 'first'),
    domains=('domain', lambda x: '; '.join(sorted(x.unique()))),
    source=('source', 'first'),
).reset_index()

print(f'Deduplicated to {len(df):,} unique NCIt codes')

## 3. Download Reference Files

Three files from NCI EVS:

- **Thesaurus.FLAT.zip** — NCIt flat file (~15 MB compressed). Tab-delimited, one concept per row.
- **nci_code_cui_map.dat** — NCIt code to UMLS CUI mapping (~4 MB).
- **SDTM Terminology txt** — CDISC-scoped preferred term lookup.

Cache behaviour: if `USE_CACHE = True` and the file exists locally at the expected minimum size, skip download.

In [ ]:
def fetch_file(url, local_path, description, min_size_kb=10):
    """Download file with caching. min_size_kb guards against cached empty files."""
    if USE_CACHE and local_path.exists():
        size_kb = local_path.stat().st_size / 1024
        if size_kb >= min_size_kb:
            print(f'  Cached: {local_path.name} ({size_kb:,.0f} KB)')
            return
        else:
            print(f'  Cached file too small ({size_kb:.1f} KB) — re-downloading')
    print(f'  Downloading {description} ...', end='', flush=True)
    r = requests.get(url, timeout=300, stream=True)
    r.raise_for_status()
    with open(local_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024 * 256):
            f.write(chunk)
    size_kb = local_path.stat().st_size / 1024
    print(f' done ({size_kb:,.0f} KB)')


fetch_file(NCIT_FLAT_URL,  NCIT_FLAT_ZIP,  'NCIt FLAT file (ZIP)', min_size_kb=5_000)
fetch_file(NCIT_CUI_URL,   NCIT_CUI_FILE,  'NCIt-to-UMLS CUI map', min_size_kb=500)
fetch_file(SDTM_URL,       SDTM_FILE,      'SDTM Terminology',     min_size_kb=500)

# Decompress FLAT zip if needed
if not NCIT_FLAT_FILE.exists() or NCIT_FLAT_FILE.stat().st_size < 1_000_000:
    print(f'  Unzipping {NCIT_FLAT_ZIP.name} ...', end='', flush=True)
    with zipfile.ZipFile(NCIT_FLAT_ZIP, 'r') as zf:
        zf.extract('Thesaurus.txt', DOWNLOADS_DIR)
    size_mb = NCIT_FLAT_FILE.stat().st_size / (1024 * 1024)
    print(f' done ({size_mb:.0f} MB)')
else:
    size_mb = NCIT_FLAT_FILE.stat().st_size / (1024 * 1024)
    print(f'  FLAT already unzipped ({size_mb:.0f} MB)')

## 4. Build NCIt Preferred Term Lookup (from SDTM Terminology)

The SDTM Terminology file is CDISC-scoped and fast to parse. Using it for the PT column
ensures we only get preferred terms that CDISC has explicitly published in its CT.

In [ ]:
sdtm_df = pd.read_csv(SDTM_FILE, sep='\t', dtype=str)
print(f'Loaded {len(sdtm_df):,} rows from {SDTM_FILE.name}')

ncit_to_pt = {}
for _, row in sdtm_df.iterrows():
    code_val = str(row.get('Code', '')).strip()
    pt_val   = str(row.get('NCI Preferred Term', '')).strip()
    if code_val and pt_val and code_val != 'nan' and pt_val != 'nan':
        ncit_to_pt[code_val] = pt_val

print(f'Built PT lookup: {len(ncit_to_pt):,} NCIt codes')

df['NCIt_PT'] = df['NCIt_code'].map(ncit_to_pt).fillna('')
n_pt = (df['NCIt_PT'] != '').sum()
print(f'PT matched: {n_pt}/{len(df)} ({n_pt/len(df)*100:.1f}%)')

## 5. Parse NCIt FLAT File

Load `Thesaurus.txt` with pandas. The FLAT file has these tab-delimited columns (per NCI ReadMe):

| Col | Content |
|---|---|
| 0 | code — NCIt C-code |
| 1 | url — full EVS URI |
| 2 | parents — pipe-delimited parent C-codes |
| 3 | synonyms — pipe-delimited; **first entry = preferred name** |
| 4 | definition — one definition (if present) |
| 5 | display_name — may be empty |
| 6 | concept_status — blank (active), Retired_Concept, Obsolete_Concept |
| 7 | semantic_type — one or more values |
| 8 | subset_membership — CDISC/NCI subsets (pipe-delimited) |

Note: NCI ReadMe documents 8 fields. The actual file has 9 — the ReadMe omits
`subset_membership` as the final column.

We extract synonyms (stripping the first entry which duplicates NCIt_PT)
and definition. Only concepts present in our CDISC CT dataset
are retained.

In [ ]:
# Column names per NCI EVS ReadMe
FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']

print(f'Reading {NCIT_FLAT_FILE.name} ...')
flat_df = pd.read_csv(
    NCIT_FLAT_FILE,
    sep='\t',
    header=None,
    names=FLAT_COLS,
    dtype=str,
    na_filter=False,   # keep empty strings as empty strings
)
print(f'Loaded {len(flat_df):,} concepts from FLAT file')

# Filter to only codes we need — keeps memory footprint small
our_codes = set(df['NCIt_code'])
flat_df = flat_df[flat_df['code'].isin(our_codes)].copy()
print(f'Filtered to {len(flat_df):,} concepts matching CDISC CT dataset')

# Build lookup dict: code -> {synonyms, definition}
# Synonyms: first pipe-entry = preferred name — strip it, keep the rest
ncit_flat = {}
for _, row in flat_df.iterrows():
    syn_list = [s.strip() for s in str(row['synonyms']).split('|') if s.strip()]
    synonyms_no_pt = syn_list[1:] if len(syn_list) > 1 else []   # drop first (= PT)
    # Deduplicate preserving order, case-insensitive
    seen_syn = set()
    deduped_syn = []
    for s in synonyms_no_pt:
        key = s.lower()
        if key not in seen_syn:
            seen_syn.add(key)
            deduped_syn.append(s)
    ncit_flat[row['code']] = {
        'synonyms':  ' | '.join(deduped_syn),
        'definition': str(row['definition']).strip(),
    }

print(f'Built FLAT lookup for {len(ncit_flat):,} codes')
print(f'  With synonyms:   {sum(1 for d in ncit_flat.values() if d["synonyms"]):,}')
print(f'  With definition: {sum(1 for d in ncit_flat.values() if d["definition"]):,}')

## 6. Process NCIt-to-UMLS CUI Mapping

The CUI map contains both Standard UMLS CUIs (C + digits) and NCI Metathesaurus
CUIs (CL + digits). These are separated into two columns in the output —
Standard CUIs are usable for cross-linking to SNOMED, MeSH, RxNorm etc.,
while NCIm CUIs are NCI-internal identifiers.

In [ ]:
cui_df = pd.read_csv(NCIT_CUI_FILE, sep='|', header=None, dtype=str,
                     on_bad_lines='skip')
cui_df = cui_df[[0, 1]].copy()
cui_df.columns = ['NCIt_code', 'CUI_value']
cui_df = cui_df.dropna()
print(f'Loaded {len(cui_df):,} NCIt -> CUI mappings')

def classify_cui(val):
    if not val or pd.isna(val):
        return 'None'
    val = str(val).strip()
    if re.match(r'^C\d+$', val):
        return 'Standard'
    if re.match(r'^CL\d+$', val):
        return 'NCI_Meta'
    return 'Other'

cui_df['CUI_type'] = cui_df['CUI_value'].apply(classify_cui)

# Build separate lookups for Standard and NCI_Meta
ncit_to_umls = {}
ncit_to_ncim = {}
for _, row in cui_df.iterrows():
    c = row['NCIt_code']
    t = row['CUI_type']
    v = row['CUI_value']
    if t == 'Standard' and c not in ncit_to_umls:
        ncit_to_umls[c] = v
    elif t == 'NCI_Meta' and c not in ncit_to_ncim:
        ncit_to_ncim[c] = v

print(f'Standard UMLS CUI: {len(ncit_to_umls):,} codes')
print(f'NCI Meta CUI:      {len(ncit_to_ncim):,} codes')

print('\nOverall CUI type distribution:')
for t, n in cui_df['CUI_type'].value_counts().items():
    print(f'  {t:12} {n:6,}')

## 7. Enrich CDISC CT DataFrame

Apply FLAT file and CUI map data to each row.

In [ ]:
rows_enriched = []

for _, row in df.iterrows():
    c = row['NCIt_code']
    flat = ncit_flat.get(c, {})

    rows_enriched.append({
        'NCIt_Code':            c,
        'TESTCD':               row['TESTCD'],
        'TEST':                 row['TEST'],
        'SDTM_Domains':         row['domains'],
        'NCIt_Preferred_Term':  row['NCIt_PT'],
        'NCIt_Synonyms':        flat.get('synonyms', ''),
        'NCIt_Definition':      flat.get('definition', ''),
        'UMLS_CUI':             ncit_to_umls.get(c, ''),
        'NCIm_CUI':             ncit_to_ncim.get(c, ''),
    })

df_out = pd.DataFrame(rows_enriched)

print(f"Enriched: {len(df_out):,} rows")
print(f"  With preferred term: {(df_out['NCIt_Preferred_Term'] != '').sum():,}")
print(f"  With synonyms:       {(df_out['NCIt_Synonyms'] != '').sum():,}")
print(f"  With definition:     {(df_out['NCIt_Definition'] != '').sum():,}")
print(f"  With UMLS CUI:       {(df_out['UMLS_CUI'] != '').sum():,}")
print(f"  With NCIm CUI:       {(df_out['NCIm_CUI'] != '').sum():,}")

## 8. Coverage Analysis

In [ ]:
print('=' * 70)
print('NCIt ENRICHMENT COVERAGE')
print('=' * 70)

n = len(df_out)

print(f"\nTotal test codes:         {n:,}")
print(f"With NCIt PT:             {(df_out['NCIt_Preferred_Term'] != '').sum():,} ({(df_out['NCIt_Preferred_Term'] != '').sum()/n*100:.1f}%)")
print(f"With synonyms:            {(df_out['NCIt_Synonyms'] != '').sum():,} ({(df_out['NCIt_Synonyms'] != '').sum()/n*100:.1f}%)")
print(f"Standard UMLS CUI:        {(df_out['UMLS_CUI'] != '').sum():,} ({(df_out['UMLS_CUI'] != '').sum()/n*100:.1f}%)  <- usable for crosslinking")
print(f"NCI Meta CUI:             {(df_out['NCIm_CUI'] != '').sum():,} ({(df_out['NCIm_CUI'] != '').sum()/n*100:.1f}%)  <- NCI-internal")
n_neither = ((df_out['UMLS_CUI'] == '') & (df_out['NCIm_CUI'] == '')).sum()
print(f"No CUI mapping at all:    {n_neither:,} ({n_neither/n*100:.1f}%)")
print(f"With definition:          {(df_out['NCIt_Definition'] != '').sum():,} ({(df_out['NCIt_Definition'] != '').sum()/n*100:.1f}%)")

print(f"\nUMLS Standard CUI coverage by domain:")
domain_total    = Counter()
domain_standard = Counter()
for _, row in df_out.iterrows():
    for dom in str(row['SDTM_Domains']).split('; '):
        dom = dom.strip()
        if dom:
            domain_total[dom] += 1
            if row['UMLS_CUI'] != '':
                domain_standard[dom] += 1
for dom in sorted(domain_total):
    tot = domain_total[dom]
    std = domain_standard.get(dom, 0)
    print(f"  {dom:6} {std:5}/{tot:5} ({std/tot*100:5.1f}%)")

## 9. Sample Inspection

Spot-check a handful of known test codes to verify enrichment quality.

In [ ]:
spot_codes = ['BASO', 'MCH', 'COTININE', 'PCT', 'PDL1S', 'GLOBUL']

for testcd in spot_codes:
    row = df_out[df_out['TESTCD'] == testcd]
    if row.empty:
        print(f"{testcd}: not found")
        continue
    r = row.iloc[0]
    print(f"\n{'─'*60}")
    print(f"TESTCD:      {r['TESTCD']}  |  NCIt: {r['NCIt_Code']}")
    print(f"PT:          {r['NCIt_Preferred_Term']}")
    print(f"Synonyms:    {r['NCIt_Synonyms']}")
    print(f"UMLS CUI:    {r['UMLS_CUI']}")
    print(f"NCIm CUI:    {r['NCIm_CUI']}")
    print(f"Definition:  {r['NCIt_Definition'][:120]}{'...' if len(r['NCIt_Definition'])>120 else ''}")

## 10. Generate SDTM_Test_Identity.xlsx

Two sheets:
- **README** — provenance, column descriptions, lookup URLs, coverage, design decisions
- **Test Codes** — one row per NCIt concept code, all enrichment columns

In [ ]:
EVS_BASE = 'https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI_Thesaurus&code='

# Styles
HDR_FONT  = Font(name='Arial', bold=True, size=10, color='FFFFFF')
HDR_FILL  = PatternFill('solid', fgColor='548235')
DATA_FONT = Font(name='Arial', size=9)
DATA_FILL = PatternFill('solid', fgColor='EBF7EE')
THIN = Border(
    left=Side(style='thin', color='AAAAAA'),
    right=Side(style='thin', color='AAAAAA'),
    top=Side(style='thin', color='AAAAAA'),
    bottom=Side(style='thin', color='AAAAAA'),
)
COLUMNS = [
    # (df_field,                header,                   width)
    ('NCIt_Code',              'NCIt_Code',               12),
    ('TESTCD',                 'TESTCD',                  12),
    ('TEST',                   'TEST',                    38),
    ('SDTM_Domains',           'SDTM_Domains',            16),
    ('NCIt_Preferred_Term',    'NCIt_Preferred_Term',     45),
    ('NCIt_Synonyms',          'NCIt_Synonyms',           55),
    ('NCIt_Definition',        'NCIt_Definition',         60),
    ('UMLS_CUI',               'UMLS_CUI',                14),
    ('NCIm_CUI',               'NCIm_CUI',                14),
]

print(f"Output columns: {len(COLUMNS)}")
for field, header, width in COLUMNS:
    print(f"  {header}")

In [ ]:
n_total    = len(df_out)
n_with_pt  = (df_out['NCIt_Preferred_Term'] != '').sum()
n_with_syn = (df_out['NCIt_Synonyms'] != '').sum()
n_with_def = (df_out['NCIt_Definition'] != '').sum()
n_umls     = (df_out['UMLS_CUI'] != '').sum()
n_ncim     = (df_out['NCIm_CUI'] != '').sum()
n_neither  = ((df_out['UMLS_CUI'] == '') & (df_out['NCIm_CUI'] == '')).sum()

# Domain legend from source data
domain_label_lookup = {}
if 'domain_label' in df_raw.columns:
    for _, row in df_raw[['domain', 'domain_label']].drop_duplicates().iterrows():
        domain_label_lookup[row['domain']] = row['domain_label']
domains_present = set()
for doms in df_out['SDTM_Domains']:
    for d in str(doms).split('; '):
        d = d.strip()
        if d:
            domains_present.add(d)
n_domains = len(domains_present)

readme_title   = Font(name='Arial', bold=True, size=14, color='548235')
readme_section = Font(name='Arial', bold=True, size=11, color='548235')
readme_body    = Font(name='Arial', size=10)
readme_bold    = Font(name='Arial', bold=True, size=10)

wb = Workbook()
ws_readme = wb.active
ws_readme.title = 'README'
ws_readme.sheet_properties.tabColor = '548235'

domain_legend_lines = []
for dom in sorted(domains_present):
    label = domain_label_lookup.get(dom, dom)
    domain_legend_lines.append((f'    {dom:4} = {label}', readme_body))

readme_lines = [
    ('SDTM TEST IDENTITY — NCIt ENRICHED REFERENCE FILE', readme_title),
    ('', None),

    ('PROVENANCE', readme_section),
    ('', None),
    (f'Generated:      {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', readme_body),
    (f'Pipeline:       SDTM_CT_Extract → SDTM_CT_NCIt_Enrich', readme_body),
    (f'SDTM CT source: NCI EVS SDTM Terminology (versionless URL, re-run to update)', readme_body),
    (f'NCIt source:    Thesaurus.FLAT.zip (NCI EVS)', readme_body),
    (f'CUI source:     nci_code_cui_map.dat (NCI EVS, updated on NCIm release schedule)', readme_body),
    ('', None),

    ('SCOPE', readme_section),
    ('', None),
    (f'All SDTM domains with TESTCD/TEST codelists ({n_domains} domains, {n_total:,} unique NCIt codes).', readme_body),
    ('One row per NCIt concept code. A concept appearing in multiple domains is listed once', readme_body),
    ('with all domain memberships in the SDTM_Domains column (semicolon-separated).', readme_body),
    ('', None),

    ('COLUMN DESCRIPTIONS', readme_section),
    ('', None),
    ('NCIt_Code              NCIt concept code — primary identifier', readme_body),
    ('TESTCD                 SDTM short test code (submission value, 8 chars max)', readme_body),
    ('TEST                   SDTM full test name (submission value)', readme_body),
    ('SDTM_Domains           Domain memberships (semicolon-separated):', readme_body),
] + domain_legend_lines + [
    ('NCIt_Preferred_Term    Preferred term from SDTM Terminology (CDISC-scoped)', readme_body),
    ('NCIt_Synonyms          Alternative names from NCIt FLAT file (pipe-separated, undifferentiated)', readme_body),
    ('NCIt_Definition        Plain-language definition from NCIt FLAT file', readme_body),
    ('UMLS_CUI               Standard UMLS CUI (C + digits) — usable for crosslinking to SNOMED, MeSH, etc.', readme_body),
    ('NCIm_CUI               NCI Metathesaurus CUI (CL + digits) — NCI-internal, not usable in UMLS', readme_body),
    ('', None),

    ('LOOKUP URLs', readme_section),
    ('', None),
    ('NCIt concept browser:  https://ncit.nci.nih.gov/ncitbrowser/ConceptReport.jsp?dictionary=NCI_Thesaurus&code={NCIt_Code}', readme_body),
    ('UMLS concept browser:  https://uts.nlm.nih.gov/uts/umls/concept/{UMLS_CUI}', readme_body),
    ('', None),

    ('COVERAGE STATISTICS', readme_section),
    ('', None),
    (f'Total test codes:              {n_total:,}', readme_bold),
    (f'With NCIt Preferred Term:      {n_with_pt:,} ({n_with_pt/n_total*100:.1f}%)', readme_body),
    (f'With synonyms:                 {n_with_syn:,} ({n_with_syn/n_total*100:.1f}%)', readme_body),
    (f'With definition:               {n_with_def:,} ({n_with_def/n_total*100:.1f}%)', readme_body),
    (f'Standard UMLS CUI:             {n_umls:,} ({n_umls/n_total*100:.1f}%)', readme_body),
    (f'NCI Meta CUI:                  {n_ncim:,} ({n_ncim/n_total*100:.1f}%)', readme_body),
    (f'No CUI mapping:                {n_neither:,} ({n_neither/n_total*100:.1f}%)', readme_body),
    ('', None),

    ('DESIGN DECISIONS', readme_section),
    ('', None),
    ('NCIt categories not included. NCIt hierarchical categories (semantic types, parent', readme_body),
    ('concepts) are too broad for test identity. "Laboratory Procedure" groups hundreds of', readme_body),
    ('unrelated tests. Tests measuring the same analyte (e.g. glucose variants) have no', readme_body),
    ('explicit relationship in NCIt hierarchy. Test identity is established through the', readme_body),
    ('NCIt concept, preferred term, synonyms, and cross-references — not categorical placement.', readme_body),
    ('', None),

    ('KNOWN LIMITATIONS', readme_section),
    ('', None),
    ('1. Synonyms are undifferentiated — the FLAT file provides no type (SY/AB/CN) or source', readme_body),
    ('   (CDISC/NCI/LOINC) annotation. The OWL format is the upgrade path if needed.', readme_body),
    ('2. UMLS CUI coverage lags NCIt releases — the CUI map follows the NCIm release schedule.', readme_body),
    ('3. This file does not include COSMoS BC/DS data — see Measurement_Specifications.xlsx.', readme_body),
]

for ri, (text, font) in enumerate(readme_lines, start=1):
    cell = ws_readme.cell(row=ri, column=1, value=text)
    if font:
        cell.font = font
ws_readme.column_dimensions['A'].width = 100

print(f'README: {len(readme_lines)} lines written')

In [ ]:
ws_data = wb.create_sheet('Test Codes')
ws_data.sheet_properties.tabColor = '548235'

# Header row
for ci, (_, header, width) in enumerate(COLUMNS, start=1):
    cell = ws_data.cell(row=1, column=ci, value=header)
    cell.font = HDR_FONT
    cell.fill = HDR_FILL
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = THIN
    ws_data.column_dimensions[get_column_letter(ci)].width = width
ws_data.row_dimensions[1].height = 28

# Data rows
for ri, (_, row) in enumerate(df_out.iterrows(), start=2):
    for ci, (field, _, _) in enumerate(COLUMNS, start=1):
        val = row.get(field, '')
        cell = ws_data.cell(row=ri, column=ci, value=str(val) if val else '')
        cell.font = DATA_FONT
        cell.fill = DATA_FILL
        cell.border = THIN
        cell.alignment = Alignment(wrap_text=True, vertical='top')

    ws_data.row_dimensions[ri].height = 30

ws_data.freeze_panes = 'A2'
ws_data.auto_filter.ref = f'A1:{get_column_letter(len(COLUMNS))}{ws_data.max_row}'

wb.save(OUTPUT_FILE)
print(f'Saved: {OUTPUT_FILE}')
print(f'  Rows: {len(df_out):,}')
print(f'  Sheets: {wb.sheetnames}')

## 11. Summary

In [ ]:
print()
print('=' * 70)
print('COMPLETE: SDTM_Test_Identity.xlsx')
print('=' * 70)
print(f'''
Output:  {OUTPUT_FILE}

Total test codes:              {len(df_out):,}
With NCIt Preferred Term:      {(df_out['NCIt_Preferred_Term'] != '').sum():,} ({(df_out['NCIt_Preferred_Term'] != '').sum()/len(df_out)*100:.1f}%)
With synonyms:                 {(df_out['NCIt_Synonyms'] != '').sum():,} ({(df_out['NCIt_Synonyms'] != '').sum()/len(df_out)*100:.1f}%)
Standard UMLS CUI:             {(df_out['UMLS_CUI'] != '').sum():,} ({(df_out['UMLS_CUI'] != '').sum()/len(df_out)*100:.1f}%)
NCI Meta CUI:                  {(df_out['NCIm_CUI'] != '').sum():,} ({(df_out['NCIm_CUI'] != '').sum()/len(df_out)*100:.1f}%)
With definition:               {(df_out['NCIt_Definition'] != '').sum():,} ({(df_out['NCIt_Definition'] != '').sum()/len(df_out)*100:.1f}%)

Enrichment sources:
  Preferred Term + Synonyms + Definition: NCI EVS (FLAT + SDTM Terminology)
  UMLS CUI + NCIm CUI:                    nci_code_cui_map.dat (NCI EVS)
''')